# Q2-question 2.2

In [1]:
from collections import Counter, defaultdict
import re

# 1) Prepare the corpus: split words into characters and add end-of-word marker "_"
def prepare_corpus(words):
    # words is a list like ["low", "low", "lowest", ...]
    corpus = []
    for w in words:
        corpus.append(list(w) + ["_"])  # e.g., "low" -> ["l","o","w","_"]
    return corpus


# 2) Count adjacent symbol pairs across the whole corpus
def count_pairs(corpus):
    pairs = Counter()
    for tokens in corpus:
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1
    return pairs


# 3) Merge the most frequent pair through the corpus
def merge_pair(corpus, pair):
    L, R = pair
    merged = []

    for tokens in corpus:
        new_tokens = []
        i = 0

        while i < len(tokens):
            # If we see L followed by R, merge them into L+R
            if i < len(tokens) - 1 and tokens[i] == L and tokens[i+1] == R:
                new_tokens.append(L + R)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        merged.append(new_tokens)

    return merged


# 4) Learn BPE merges and return both the merges and the evolving vocabulary
def learn_bpe(corpus, num_merges=10, verbose=True):
    merges = []
    vocab_evolution = []

    for step in range(1, num_merges + 1):
        pairs = count_pairs(corpus)

        if not pairs:
            break

        (L, R), freq = pairs.most_common(1)[0]
        merges.append((L, R, freq))
        corpus = merge_pair(corpus, (L, R))

        # Track vocabulary after each merge
        vocab = sorted({sym for tokens in corpus for sym in tokens})
        vocab_evolution.append(vocab)

        if verbose:
            print(f"\nStep {step}: merge {L} + {R} -> {L+R} (freq={freq})")
            print("Sample corpus row:", " ".join(corpus[0][:8]))
            print("Vocab size:", len(vocab), "| Vocab head:", vocab[:15])

    return merges, vocab_evolution, corpus


# 5) A simple greedy segmenter that applies the learned merges to new words
# (This mirrors the idea of "longest-match using merge ranks")
def build_merge_ranks(merges):
    return {
        (L + R if isinstance(L, list) else L, R): i
        for i, (L, R, freq) in enumerate(merges)
    }


def segment_word_bpe(word, merge_ranks):
    # Start from characters + "_"
    tokens = list(word) + ["_"]

    # Keep trying to merge best-ranked adjacent pair
    while True:
        pairs = [
            ((tokens[i], tokens[i+1]), i)
            for i in range(len(tokens) - 1)
        ]

        # Pick the pair that appears in merge_ranks with smallest rank
        ranked = [
            (merge_ranks[p], idx)
            for p, idx in pairs
            if p in merge_ranks
        ]

        if not ranked:
            break

        _, best_idx = min(ranked, key=lambda x: x[0])

        # Merge the best pair at best_idx
        L, R = tokens[best_idx], tokens[best_idx + 1]
        tokens = (
            tokens[:best_idx]
            + [L + R]
            + tokens[best_idx + 2:]
        )

    return tokens


def segment_sentence(sentence, merge_ranks):
    return [segment_word_bpe(w, merge_ranks) for w in sentence.split()]


In [2]:
# ---- Run the mini-BPE demo ----

text = "low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new"
words = text.split()

# (A) Show initial character-level corpus and vocabulary
corpus = prepare_corpus(words)
initial_vocab = sorted({sym for row in corpus for sym in row})
print("Initial vocabulary:", initial_vocab)

# (B) Learn a small number of merges, printing each step like the slides
merges, vocab_evolution, final_corpus = learn_bpe(
    corpus, num_merges=10, verbose=True
)

# (C) Build a segmenter from the learned merges and try it on a few words
merge_ranks = build_merge_ranks(merges)
for w in ["new", "newer", "lowest", "widest", "newestnew"]:
    print(f"{w:8} ->", " ".join(segment_word_bpe(w, merge_ranks)))

Initial vocabulary: ['_', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']

Step 1: merge e + r -> er (freq=9)
Sample corpus row: l o w _
Vocab size: 11 | Vocab head: ['_', 'd', 'e', 'er', 'i', 'l', 'n', 'o', 's', 't', 'w']

Step 2: merge er + _ -> er_ (freq=9)
Sample corpus row: l o w _
Vocab size: 11 | Vocab head: ['_', 'd', 'e', 'er_', 'i', 'l', 'n', 'o', 's', 't', 'w']

Step 3: merge n + e -> ne (freq=8)
Sample corpus row: l o w _
Vocab size: 11 | Vocab head: ['_', 'd', 'e', 'er_', 'i', 'l', 'ne', 'o', 's', 't', 'w']

Step 4: merge ne + w -> new (freq=8)
Sample corpus row: l o w _
Vocab size: 11 | Vocab head: ['_', 'd', 'e', 'er_', 'i', 'l', 'new', 'o', 's', 't', 'w']

Step 5: merge l + o -> lo (freq=7)
Sample corpus row: lo w _
Vocab size: 10 | Vocab head: ['_', 'd', 'e', 'er_', 'i', 'lo', 'new', 's', 't', 'w']

Step 6: merge lo + w -> low (freq=7)
Sample corpus row: low _
Vocab size: 10 | Vocab head: ['_', 'd', 'e', 'er_', 'i', 'low', 'new', 's', 't', 'w']

Step 7: merge new + e

BPE uses subword tokens to reduce the out-of-vocabulary (OOV) problem by representing unknown words as combinations of known subwords. Instead of requiring every complete word to appear in the training vocabulary, BPE can break rare or unseen words into smaller units. For example, an unseen word such as newestnew can be segmented into known subwords such as new, e, s, t, new, and \_. This allows the model to process new or rare words even when the complete word was not seen during training. Some learned subwords can also correspond to meaningful morphemes; for example, er_ can represent the English suffix “-er,” which can indicate an agent noun (teacher) or a comparative form (faster). Therefore, BPE provides better coverage of rare and unseen words while keeping the vocabulary more manageable.

# Q2-question 2.3

In [3]:
# 6) Get five most frequent merges
def get_top_merges(merges, n=5):
    # Sort all learned merges by frequency
    top_merges = sorted(
        merges,
        key=lambda x: x[2],
        reverse=True
    )[:n]

    print(f"\n=== Top {n} Most Frequent Merges ===")

    for i, (L, R, freq) in enumerate(top_merges, 1):
        print(
            f"{i}. '{L}' + '{R}' -> "
            f"'{L+R}' (frequency: {freq})"
        )

    return top_merges

# 7) Get five longest subword tokens
def get_longest_tokens(vocab, n=5):
    # Filter out single characters and end marker
    tokens = [t for t in vocab if len(t) > 1 and t != '_']
    # Sort by length from longest to shortest
    longest = sorted(tokens, key=len, reverse=True)[:n]
    
    print(f"\n=== Top {n} Longest Subword Tokens ===")
    
    for i, token in enumerate(longest, 1):
        print(f"{i}. '{token}' (length: {len(token)})")
        
    return longest

# 8) Segment selected words
def segment_custom_words(words, merge_ranks):
    print("\n=== Word Segmentation Results ===")
    
    for word in words:
        segmented = segment_word_bpe(word, merge_ranks)
        print(f"'{word:20}' -> {' | '.join(segmented)}")

In [4]:
# ---- Run the 2.3 question ----

text = '''我喜欢学习自然语言处理。
自然语言处理可以帮助计算机理解人类语言。
我正在学习机器学习和人工智能。
这些技术可以用于文本分类、机器翻译和问答系统。
通过学习，我可以理解子词分词是如何工作的。'''

# Prepare a Chinese word list from the paragraph
words = [
    "我", "喜欢", "学习", "自然语言处理",
    "自然语言处理", "可以", "帮助", "计算机",
    "理解", "人类语言", "我", "正在", "学习",
    "机器学习", "人工智能", "这些", "技术",
    "可以", "用于", "文本分类", "机器翻译",
    "问答系统", "通过", "学习", "我",
    "可以", "理解", "子词分词", "如何", "工作"
]

print("=" * 60)
print("BPE TOKENIZATION ANALYSIS")
print("=" * 60)


# (A) Show initial character-level corpus and vocabulary

corpus = prepare_corpus(words)

initial_vocab = sorted({
    sym
    for row in corpus
    for sym in row
})

print(f"\nInitial vocabulary size: {len(initial_vocab)}")
print("Initial vocabulary:", initial_vocab)


# (B) Learn 30 BPE merges

print("\n" + "=" * 60)
print("LEARNING BPE MERGES")
print("=" * 60)

merges, vocab_evolution, final_corpus = learn_bpe(
    corpus,
    num_merges=30,
    verbose=True
)

print(
    f"\nTotal number of merges learned: "
    f"{len(merges)}"
)

# (C) Show 5 most frequent merges

top_merges = get_top_merges(
    merges,
    n=5
)


# (D) Show 5 longest subword tokens

final_vocab = (
    vocab_evolution[-1]
    if vocab_evolution
    else initial_vocab
)

longest_tokens = get_longest_tokens(
    final_vocab,
    n=5
)


# (E) Build tokenizer

merge_ranks = build_merge_ranks(merges)


# Select 5 different words from the paragraph
# including a relatively rare word and a derived/compound word

custom_words = [
    "学习",
    "自然语言处理",
    "计算机",
    "子词分词",
    "机器学习"
]

print("\n" + "=" * 60)
print("WORD SEGMENTATION")
print("=" * 60)

segment_custom_words(
    custom_words,
    merge_ranks
)


# Additional examples

print("\n=== Additional Word Segmentations ===")

for w in [
    "喜欢",
    "帮助",
    "人工智能",
    "文本分类",
    "机器翻译",
    "问答系统"
]:
    seg = segment_word_bpe(
        w,
        merge_ranks
    )

    print(
        f"'{w:15}' -> {' | '.join(seg)}"
    )


BPE TOKENIZATION ANALYSIS

Initial vocabulary size: 50
Initial vocabulary: ['_', '习', '于', '些', '人', '以', '何', '作', '分', '助', '可', '喜', '器', '在', '处', '如', '子', '学', '工', '帮', '我', '技', '文', '智', '本', '术', '机', '欢', '正', '然', '理', '用', '答', '算', '类', '系', '统', '翻', '能', '自', '解', '言', '计', '词', '译', '语', '过', '这', '通', '问']

LEARNING BPE MERGES

Step 1: merge 学 + 习 -> 学习 (freq=4)
Sample corpus row: 我 _
Vocab size: 49 | Vocab head: ['_', '于', '些', '人', '以', '何', '作', '分', '助', '可', '喜', '器', '在', '处', '如']

Step 2: merge 学习 + _ -> 学习_ (freq=4)
Sample corpus row: 我 _
Vocab size: 49 | Vocab head: ['_', '于', '些', '人', '以', '何', '作', '分', '助', '可', '喜', '器', '在', '处', '如']

Step 3: merge 我 + _ -> 我_ (freq=3)
Sample corpus row: 我_
Vocab size: 49 | Vocab head: ['_', '于', '些', '人', '以', '何', '作', '分', '助', '可', '喜', '器', '在', '处', '如']

Step 4: merge 语 + 言 -> 语言 (freq=3)
Sample corpus row: 我_
Vocab size: 48 | Vocab head: ['_', '于', '些', '人', '以', '何', '作', '分', '助', '可', '喜', '器', '在', '处', '如

The BPE model learned different types of subwords, including individual Chinese characters, common character combinations, parts of words, and longer whole-word units. For example, common expressions such as “自然语言处理” and “机器学习” were learned as longer subword tokens when their character combinations occurred frequently. For Chinese, BPE mainly learns frequent character combinations and word units rather than clearly separated prefixes, suffixes, and stems as in English. One advantage of subword tokenization is that it can handle rare or unseen words by breaking them into smaller subwords that the model already knows. Another advantage is that BPE can preserve common character combinations while keeping the vocabulary smaller than a word-level tokenizer. One disadvantage is that some Chinese words may be split into unnatural subwords, which can make it harder for the model to understand their complete meaning. Overall, BPE provides a useful balance between character-level and word-level tokenization for Chinese.


# Q5-question 1

In [5]:
text = '''我喜欢学习自然语言处理。自然语言处理可以帮助计算机理解人类语言。我正在学习机器学习和人工智能。这些技术可以用于文本分类、机器翻译和问答系统。通过学习，我可以理解子词分词是如何工作的。'''

# Naïve space-based tokenization
tokens = text.split()

print(tokens)

['我喜欢学习自然语言处理。自然语言处理可以帮助计算机理解人类语言。我正在学习机器学习和人工智能。这些技术可以用于文本分类、机器翻译和问答系统。通过学习，我可以理解子词分词是如何工作的。']


Manually corrected version:
[
'我', '喜', '欢', '学', '习', '自', '然', '语', '言', '处', '理', '。',
'自', '然', '语', '言', '处', '理', '可', '以', '帮', '助', '计', '算', '机', '理', '解', '人', '类', '语', '言', '。',
'我', '正', '在', '学', '习', '机', '器', '学', '习', '和', '人', '工', '智', '能', '。',
'这', '些', '技', '术', '可', '以', '用', '于', '文', '本', '分', '类', '、', '机', '器', '翻', '译', '和', '问', '答', '系', '统', '。',
'通', '过', '学', '习', '，', '我', '可', '以', '理', '解', '子', '词', '分', '词', '是', '如', '何', '工', '作', '的', '。'
]

In Chinese, a naïve space-based tokenizer treats this Chinese text as one token because Chinese text usually does not use spaces between words. I manually corrected the tokens by separating Chinese characters and punctuation marks. Chinese does not use clitics in the same way as English, and suffixes are not separated in this simple character-level tokenization. The main difference is that the naïve tokenizer produces one large token, while the manually corrected version produces separate character and punctuation tokens. Alternatively, the text can also be tokenized into Chinese words depending on the desired tokenization method.

# Q5-question 2

In [6]:
# Question 2
import jieba

text = '''我喜欢学习自然语言处理。自然语言处理可以帮助计算机理解人类语言。我正在学习机器学习和人工智能。这些技术可以用于文本分类、机器翻译和问答系统。通过学习，我可以理解子词分词是如何工作的。'''

# Tokenization using jieba
tool_tokens = list(jieba.cut(text))

print(tool_tokens)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ruyi4\AppData\Local\Temp\jieba.cache
Loading model cost 0.977 seconds.
Prefix dict has been built successfully.


['我', '喜欢', '学习', '自然语言', '处理', '。', '自然语言', '处理', '可以', '帮助', '计算机', '理解', '人类', '语言', '。', '我', '正在', '学习', '机器', '学习', '和', '人工智能', '。', '这些', '技术', '可以', '用于', '文本', '分类', '、', '机器翻译', '和', '问答', '系统', '。', '通过', '学习', '，', '我', '可以', '理解', '子词', '分词', '是', '如何', '工作', '的', '。']


- My manual tokenization uses character-level tokenization, where each Chinese character is treated as a separate token:
我 | 喜 | 欢 | 学 | 习 | 自 | 然 | 语 | 言 | 处 | 理 | 。|...
- The jieba tokenizer instead groups characters into Chinese words:
我 | 喜欢 | 学习 | 自然语言 | 处理 | 。|...

For example, 喜 + 欢 becomes 喜欢, and 学 + 习 becomes 学习. Similarly, 自然语言 and other common word combinations are treated as single tokens by the tool.

The differences occur because my manual tokenization uses character-level tokenization, while jieba uses word-level tokenization. Therefore, the two methods have different tokenization granularity.

# Q5-question 3

In Chinese, some multiword expressions can be treated as single tokens because they represent a fixed concept or have a specific meaning as a whole.

- 北京 (Beijing): This is a place name. It refers to a specific geographic location, so it can be treated as a single token.
- 心想事成 (May all your wishes come true): This is a Chinese idiom with a fixed meaning. Its overall meaning cannot be understood simply by treating each character as an independent token, so it can be treated as a single token.
- 自然语言处理 (Natural Language Processing): This is a fixed technical term in computer science. The complete expression refers to a specific field, so treating it as a single token helps preserve its meaning.

These expressions should be treated as single tokens because their combined meanings are more specific than the meanings of their individual parts. Keeping them as single tokens can help an NLP system better preserve their semantic information.

# Q5-question 4

The hardest part of tokenization in Chinese is identifying word boundaries because Chinese text usually does not use spaces between words. For example, it can be difficult to decide whether “自然语言处理” should be treated as one token or several tokens. Compared with English, Chinese tokenization is more difficult because English usually uses spaces to separate words. Punctuation also makes tokenization more challenging because the tokenizer needs to decide how punctuation should be handled; for example, a number such as 1.23 should usually be treated as one complete token rather than split into 1, ., and 23. Morphology is somewhat different because Chinese has fewer word-form changes than English, although Chinese still needs to handle linguistic structures such as the difference between “学生” and “学生们” for singular and plural meanings. MWEs can also make tokenization more difficult because expressions such as “自然语言处理” have a specific meaning as a whole and may be better treated as a single token.